<div style="border-left: 5px solid #b7791f; background-color: #fff8e1; padding: 0.8em 1em; margin: 1em 0; border-radius: 4px;">
  <strong>Warning: AI-assisted materials</strong><br><br>
  These materials were developed with assistance from AI tools. All content has been reviewed and edited by the instructor, who takes final responsibility for its accuracy, clarity, and appropriateness for the course. Students should treat these materials as instructor-reviewed course content while applying the same critical judgment they would use with any technical material. Please report any suspected errors or unclear explanations to ghunt@wm.edu.
</div>

## Curse of Dimensionality

KNN is based on a simple idea:
- to predict at $x$, look at points near $x$
- average their responses (regression) or take a majority vote (classification)

So everything depends on the notion of **distance** and **local neighborhoods**.

We sometimes run into problems when the dimension $D$ of the input $x$-space  gets too large. This often called **the curse of dimensionality**.

What are some features of the curse?

**Distances become less informative**

In low dimensions:
- nearby points are meaningfully closer than far-away points

In high dimensions:
- distances between points tend to become similar
- the gap between the nearest and farthest neighbor shrinks

So:
- the “nearest” neighbor may not be meaningfully close
- distance no longer clearly separates relevant vs irrelevant points

**Neighborhoods are no longer local**

Recall the goal:
$$
s^*(x)=\mathbb{E}[Y\mid X=x]
$$

KNN approximates this by averaging nearby points.

But in high dimensions:
- data becomes very sparse
- there may be very few points truly close to $x$

To get $k$ neighbors, we are forced to look farther away:
- the neighborhood grows large

So the method is no longer truly “local”.

**Sample size requirements explode**

A useful intuition:

- In 1D, covering an interval requires a modest number of points  
- In $D$ dimensions, covering space requires exponentially many points  

So to maintain the same notion of “locality”:
- the required sample size grows very quickly with $D$

With limited data:
- KNN cannot reliably approximate $\mathbb{E}[Y\mid X=x]$

We can simulate how the distances between points typically change as a function of the dimension

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import pairwise_distances

In [ ]:
def simulate_distance_concentration(N=500, Ds=(1, 2, 5, 10, 20, 50, 100), seed=0):
    rng = np.random.default_rng(seed)

    mean_nearest = []
    mean_farthest = []
    mean_ratio = []

    for D in Ds:
        # Sample N points uniformly from [0,1]^D
        X = rng.uniform(0, 1, size=(N, D))

        # Pairwise Euclidean distances
        dist = pairwise_distances(X, metric="euclidean")

        # Ignore self-distances on diagonal when computing nearest neighbor
        np.fill_diagonal(dist, np.inf)
        nearest = dist.min(axis=1)

        # Restore diagonal to 0 so farthest neighbor is computed correctly
        np.fill_diagonal(dist, 0.0)
        farthest = dist.max(axis=1)

        mean_nearest.append(nearest.mean())
        mean_farthest.append(farthest.mean())
        mean_ratio.append((nearest / farthest).mean())

    return np.array(Ds), np.array(mean_nearest), np.array(mean_farthest), np.array(mean_ratio)

In [ ]:
Ds, mean_nearest, mean_farthest, mean_ratio = simulate_distance_concentration(seed=None)

In [ ]:
plt.figure(figsize=(7, 5))
plt.plot(Ds, mean_nearest, marker='o', label="mean nearest-neighbor distance")
plt.plot(Ds, mean_farthest, marker='o', label="mean farthest-neighbor distance")
plt.xlabel("dimension D")
plt.ylabel("distance")
plt.title("Distances in [0,1]^D")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(7, 5))
plt.plot(Ds, mean_ratio, marker='o')
plt.xlabel("dimension D")
plt.ylabel("mean(nearest / farthest)")
plt.title("Nearest and farthest distances become similar")
plt.show()

We can also simulate density as we change the dimension

In [ ]:
def simulate_local_mass(N=10000, Ds=(1, 2, 5, 10, 20, 50), r=0.2, seed=None):
    rng = np.random.default_rng(seed)
    avg_counts = []

    for D in Ds:
        X = rng.uniform(0, 1, size=(N, D))
        x0 = np.full(D, 0.5)  # center point to avoid edge effects
        dists = np.sqrt(((X - x0) ** 2).sum(axis=1))
        avg_counts.append((dists <= r).mean())

    return np.array(Ds), np.array(avg_counts)

In [ ]:
Ds, counts = simulate_local_mass()

In [ ]:
plt.figure(figsize=(7, 5))
plt.plot(Ds, counts, marker='o')
plt.xlabel("dimension D")
plt.ylabel("pct of points within radius r")
plt.title("Local neighborhoods become empty in high dimensions")
plt.show()

### Consequences for KNN

Sometimes this means that KNN works best in low to moderate dimensions. However, the curse of dimensionality is a worst-case phenomenon. In many real datasets data may lie near a lower-dimensional structure, and so distances aren't that meaningless. 

**Note** that KNN also has an inability to ignore irrelevant features, though this is slightly different from the curse of dimensionality. 

Lets do a simulation changing the dimension $D$:

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

In [ ]:
def simulate_models(
    f,
    Ds=(1, 2, 5, 10, 20, 50),
    N=2000,
    noise=0.1,
    k=10,
    test_size=0.3,
    seed=None,
):
    rng = np.random.default_rng(seed)

    knn_mses = []
    lin_mses = []
    radii = []

    for D in Ds:
        X = rng.uniform(0, 1, size=(N, D))

        # true function passed in
        f_vals = f(X)

        y = f_vals + noise * rng.standard_normal(N)

        print(f"SNR={np.std(f_vals)/noise}")

        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=test_size, random_state=seed
        )

        # fit KNN
        knn = KNeighborsRegressor(n_neighbors=k)
        knn.fit(X_train, y_train)
        y_knn = knn.predict(X_test)
        knn_mses.append(mean_squared_error(y_test, y_knn))

        # fit Linear regression
        lin = LinearRegression()
        lin.fit(X_train, y_train)
        y_lin = lin.predict(X_test)
        lin_mses.append(mean_squared_error(y_test, y_lin))

    return np.array(Ds), np.array(knn_mses), np.array(lin_mses)

Let's consider two "true" models. One linear and one non-linear. Here we divide by $\sqrt{D}$ to try to make sure the amount of signal stays constant as the dimension changes.

In [ ]:
def f_linear(X):
    D = X.shape[1]
    return np.sqrt(6) * (1 / np.sqrt(D)) * (X - 0.5).sum(axis=1)

def f_nonlinear(X):
    D = X.shape[1]
    return (1 / np.sqrt(D)) * np.sin(2 * np.pi * X).sum(axis=1)

In [ ]:
k_choose = 5
N_choose = 500
Ds, knn_nl, lin_nl = simulate_models(f_nonlinear,k=k_choose, N=N_choose)

In [ ]:
Ds, knn_lin, lin_lin = simulate_models(f_linear,k=k_choose, N=N_choose)

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(Ds, knn_nl, marker='o', label="KNN (nonlinear)")
plt.plot(Ds, lin_nl, marker='o', label="Linear (nonlinear)")
plt.xlabel("dimension D")
plt.ylabel("test MSE")
plt.title("Nonlinear target")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(Ds, knn_lin, marker='o', label="KNN (linear)")
plt.plot(Ds, lin_lin, marker='o', label="Linear (linear)")
plt.xlabel("dimension D")
plt.ylabel("test MSE")
plt.title("Linear target")
plt.legend()
plt.show()